**The Euler-Cromer method: fixing energy drift in oscillatory simulations.**

In the previous notebook (`euler_pendulum.ipynb`) we saw that Forward Euler
causes the pendulum amplitude to grow continuously.
This is not a numerical rounding error - it is a structural flaw:
Forward Euler systematically *adds* energy to oscillatory systems at every step.

**Why Forward Euler fails for oscillators.**
At each step, Forward Euler updates $\theta$ using the *old* velocity
before $\omega$ has been corrected by the current force:

$$\omega_{i+1} = \omega_i - \frac{g}{L}\sin\theta_i\,\Delta t
\qquad
\theta_{i+1} = \theta_i + \omega_{\mathbf{i}}\,\Delta t$$

Because it uses stale velocity information, the position overshoots
slightly on every swing, injecting a small amount of extra energy each cycle.
Over many cycles this accumulates and the amplitude grows without bound.

**Phase space and why it matters.**
For any conservative system (one with no friction or driving force),
the **total mechanical energy** $E = \frac{1}{2}\omega^2 - \frac{g}{L}\cos\theta$
is constant.
If we plot $\omega$ vs $\theta$ - called a **phase space** diagram -
the exact trajectory is a *closed oval*: the system returns to exactly
the same state after every period.
Forward Euler's trajectory in phase space is a *spiral that drifts outward*
because energy keeps growing.
A good integrator should trace a closed loop.

**Cromer's fix - one word changed.**
Harvey Cromer (1981) noticed that swapping just one index eliminates most
of the drift.
Instead of updating $\theta$ with the old $\omega_i$, use the
**newly computed** $\omega_{i+1}$:

$$\omega_{i+1} = \omega_i - \frac{g}{L}\sin\theta_i\,\Delta t
\qquad
\theta_{i+1} = \theta_i + \omega_{\mathbf{i+1}}\,\Delta t$$

This single change makes the method **symplectic** - it preserves a
discrete analog of energy, so the phase-space trajectory stays near
the correct closed loop rather than spiraling away.
The energy still oscillates slightly around the true value
(it does not drift monotonically), which is a hallmark of symplectic integrators.
Session 19 will explore this more deeply and introduce higher-order
symplectic methods (Velocity Verlet, Yoshida 4th-order) that keep the
orbit even closer to the exact contour.

---
**Simulation parameters:**

Same grid as the Euler notebook: 10 seconds, 500 time steps, with

$$
\Delta t = \frac{10\ \text{s}}{500} = 0.02\ \text{s} = 20\ \text{ms}.
$$

**Initial conditions:**

The pendulum is released from rest with an initial angular displacement

$$
\theta_0 = 45^\circ,
\qquad
\omega_0 = 0,
$$

matching the Euler notebook so the results can be compared directly.

**Total mechanical energy**

Energy is measured relative to the lowest point of the swing
($\theta = 0$).

The kinetic energy is

$$
K = \frac{1}{2} m L^2 \omega^2.
$$

The potential energy is

$$
U = mgL\left(1 - \cos\theta\right).
$$

The total mechanical energy is therefore

$$
E = K + U
= \frac{1}{2} m L^2 \omega^2
+ mgL\left(1 - \cos\theta\right).
$$

For the initial conditions,

$$
E_0
=
\frac{1}{2} m L^2 \omega_0^2
+
mgL\left(1-\cos\theta_0\right).
$$

Because the Euler-Cromer method is approximately energy-conserving,
the total energy should remain close to \(E_0\) throughout the simulation.

In [ ]:
"""cromer_pendulum.ipynb"""

# Cell 01 - Simulation parameters

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

tf = 10  # final time (seconds)
ts = 500  # number of time steps
dt = tf / ts  # time step size (0.02 s = 20 ms)

mass = 1.0  # pendulum bob mass (kg)
length = 1.0  # pendulum length (m)
g = 9.81  # gravitational acceleration (m/s^2)

t = np.zeros(ts)  # Array for time (seconds)
omega = np.zeros(ts)  # Array for angular velocity (rad/s)
theta = np.zeros(ts)  # Array for angle (radians)
theta[0] = np.deg2rad(45)  # initial displacement (released from rest)

print(f"{tf=:,}  {ts=:,}  {dt=:.3f}")

---
**The Euler-Cromer integration loop**
The only difference from Forward Euler is highlighted in the comment:
`omega[i+1]` (the freshly updated velocity) is used instead of `omega[i]`
when stepping $\theta$ forward


In [ ]:
# Cell 02 - Euler-Cromer integration
# The ONLY change from Forward Euler: theta uses omega[i+1], not omega[i]

for i in range(ts - 1):
    t[i + 1] = t[i] + dt
    omega[i + 1] = omega[i] - g / length * np.sin(theta[i]) * dt
    theta[i + 1] = theta[i] + omega[i + 1] * dt  # <-- Cromer's fix

print(f"Final values: {t[-1]:.3f} secs,  {omega[-1]:.3f} rad/s, {theta[-1]:.3f} rad")

---
**Plotting $\theta$ and $\omega$.**
Unlike the Forward Euler result, the amplitude here stays approximately
constant over the full 10 seconds.
The slight oscillation in amplitude (rather than monotonic growth) is
characteristic of a symplectic integrator - energy wobbles around
the true value instead of drifting away from it.

In [ ]:
# Cell 03 - Dual y-axis plot of angular displacement and angular velocity

fig, ax1 = plt.subplots(figsize=(9, 5))

(plot1,) = ax1.plot(t, theta, lw=2, label=r"$\theta$ (rad)")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel(r"Angular Displacement $\theta$ (rad)")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
(plot2,) = ax2.plot(t, omega, lw=2, color="orange", label=r"$\omega$ (rad/s)")
ax2.set_ylabel(r"Angular Velocity $\omega$ (rad/s)")

plt.title("Simple Pendulum - Euler-Cromer Method")
plt.legend(
    [plot1, plot2], [r"$\theta$", r"$\omega$"], framealpha=1.0, facecolor="white"
)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 04 - Energy analysis

# Initial Kinetic energy (should be zero at t=0)
Ei = 0.5 * mass * length**2 * omega[0] ** 2
# Initial Potential energy (relative to lowest point of the swing)
Ei += mass * g * length * (1 - np.cos(theta[0]))

# Final Kinetic energy
Ef = 0.5 * mass * length**2 * omega[-1] ** 2
# Final Potential energy
Ef += mass * g * length * (1 - np.cos(theta[-1]))

# Energy drift in joules as a percentage of initial energy
drift_pct = 100 * (Ef - Ei) / Ei

print(f"Initial energy : {Ei:6.3f} J")
print(f"Final energy   : {Ef:6.3f} J")
print(f"Energy drift   : {Ef - Ei:6.3f} J  ({drift_pct:6.3f}%)")